In [1]:
from __future__ import annotations

import argparse
import ast
import html
import json
import re
from pathlib import Path

import pandas as pd


def resolve_project_root() -> Path:
    """Locate the repository root from the notebook's current working directory."""

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "raw_data").exists() or (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the CineIQ project root from the current working directory. "
        f"Started from: {cwd}. Open the notebook from inside the cloned repository so "
        "raw_data, cleaned_data, and model_artifacts stay inside the project."
    )


PROJECT_ROOT = resolve_project_root()
DEFAULT_BASE = PROJECT_ROOT / "raw_data"
DEFAULT_ML25M = DEFAULT_BASE / "ml-25m"
DEFAULT_OUTPUT = PROJECT_ROOT / "cleaned_data"


def clean_text(value: object) -> str:
    """Normalize review/tag/title text while preserving useful words."""
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_json_list(value: object) -> list[dict]:
    """Safely parse TMDB JSON-ish list columns."""
    if pd.isna(value) or value == "":
        return []
    try:
        parsed = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []


def names_from_people(people: list[dict], limit: int | None = None) -> list[str]:
    names = [str(person.get("name", "")).strip() for person in people]
    names = [name for name in names if name]
    return names[:limit] if limit else names


def jobs_from_crew(crew: list[dict], jobs: set[str], limit: int | None = None) -> list[str]:
    names = [
        str(person.get("name", "")).strip()
        for person in crew
        if str(person.get("job", "")).strip() in jobs
    ]
    names = [name for name in names if name]
    return names[:limit] if limit else names


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Wrote {path} ({len(df):,} rows)")


def clean_movies(movies_path: Path, output_dir: Path) -> pd.DataFrame:
    movies = pd.read_csv(movies_path)
    movies = movies.drop_duplicates(subset=["movieId"]).copy()

    movies["title"] = movies["title"].map(clean_text)
    movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$")[0]
    movies["clean_title"] = movies["title"].str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
    movies["genres"] = movies["genres"].fillna("")
    movies["genres_list"] = movies["genres"].replace("(no genres listed)", "").str.replace("|", " ", regex=False)
    movies["primary_genre"] = movies["genres"].apply(
        lambda x: "" if x == "(no genres listed)" else str(x).split("|")[0]
    )
    movies["genre_count"] = movies["genres"].apply(
        lambda x: 0 if x == "(no genres listed)" or pd.isna(x) else len(str(x).split("|"))
    )
    movies["year"] = pd.to_numeric(movies["year"], errors="coerce").astype("Int64")

    keep = ["movieId", "title", "clean_title", "year", "genres", "genres_list", "primary_genre", "genre_count"]
    movies = movies[keep].sort_values("movieId")
    write_csv(movies, output_dir / "movies_clean.csv")
    return movies


def clean_links(links_path: Path, output_dir: Path) -> pd.DataFrame:
    links = pd.read_csv(links_path)
    links = links.drop_duplicates(subset=["movieId"]).copy()
    links["imdbId"] = pd.to_numeric(links["imdbId"], errors="coerce").astype("Int64")
    links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce").astype("Int64")
    links["imdb_url_id"] = links["imdbId"].apply(lambda x: f"tt{int(x):07d}" if pd.notna(x) else "")
    write_csv(links, output_dir / "links_clean.csv")
    return links


def clean_imdb_reviews(imdb_path: Path, output_dir: Path, sample_rows: int | None) -> pd.DataFrame:
    reviews = pd.read_csv(imdb_path, nrows=sample_rows)
    reviews = reviews.drop_duplicates().dropna(subset=["review", "sentiment"]).copy()
    reviews["review_clean"] = reviews["review"].map(clean_text)
    reviews["sentiment"] = reviews["sentiment"].str.lower().str.strip()
    reviews = reviews[reviews["sentiment"].isin(["positive", "negative"])]
    reviews["sentiment_label"] = reviews["sentiment"].map({"negative": 0, "positive": 1})
    reviews["review_length"] = reviews["review_clean"].str.split().str.len()
    reviews = reviews[reviews["review_length"] > 2]
    reviews = reviews[["review_clean", "sentiment", "sentiment_label", "review_length"]]
    write_csv(reviews, output_dir / "imdb_reviews_clean.csv")
    return reviews


def clean_tmdb_credits(tmdb_path: Path, output_dir: Path, sample_rows: int | None) -> pd.DataFrame:
    credits = pd.read_csv(tmdb_path, nrows=sample_rows)
    credits = credits.drop_duplicates(subset=["movie_id"]).copy()
    credits["title"] = credits["title"].map(clean_text)

    cast_lists = credits["cast"].apply(parse_json_list)
    crew_lists = credits["crew"].apply(parse_json_list)

    credits["top_cast"] = cast_lists.apply(lambda people: "|".join(names_from_people(people, 5)))
    credits["cast_count"] = cast_lists.apply(len)
    credits["director"] = crew_lists.apply(lambda crew: "|".join(jobs_from_crew(crew, {"Director"})))
    credits["writers"] = crew_lists.apply(
        lambda crew: "|".join(jobs_from_crew(crew, {"Writer", "Screenplay", "Story"}, 5))
    )
    credits["producers"] = crew_lists.apply(lambda crew: "|".join(jobs_from_crew(crew, {"Producer"}, 5)))
    credits["composer"] = crew_lists.apply(lambda crew: "|".join(jobs_from_crew(crew, {"Original Music Composer"}, 3)))

    keep = ["movie_id", "title", "top_cast", "cast_count", "director", "writers", "producers", "composer"]
    credits = credits[keep].rename(columns={"movie_id": "tmdbId", "title": "tmdb_title"})
    credits["tmdbId"] = pd.to_numeric(credits["tmdbId"], errors="coerce").astype("Int64")
    write_csv(credits, output_dir / "tmdb_credits_clean.csv")
    return credits


def clean_ratings(
    ratings_path: Path,
    output_dir: Path,
    sample_rows: int | None,
    chunksize: int,
    write_full_ratings: bool,
) -> pd.DataFrame:
    stats_parts = []
    cleaned_path = output_dir / "ratings_clean.csv"
    if write_full_ratings and cleaned_path.exists():
        cleaned_path.unlink()

    rows_seen = 0
    reader = pd.read_csv(ratings_path, chunksize=chunksize)
    for chunk_no, chunk in enumerate(reader, start=1):
        if sample_rows is not None:
            remaining = sample_rows - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)

        chunk = chunk.dropna(subset=["userId", "movieId", "rating", "timestamp"]).copy()
        chunk["userId"] = pd.to_numeric(chunk["userId"], errors="coerce").astype("Int64")
        chunk["movieId"] = pd.to_numeric(chunk["movieId"], errors="coerce").astype("Int64")
        chunk["rating"] = pd.to_numeric(chunk["rating"], errors="coerce")
        chunk["timestamp"] = pd.to_numeric(chunk["timestamp"], errors="coerce").astype("Int64")
        chunk = chunk.dropna(subset=["userId", "movieId", "rating", "timestamp"])
        chunk = chunk[(chunk["rating"] >= 0.5) & (chunk["rating"] <= 5.0)]
        chunk = chunk.drop_duplicates(subset=["userId", "movieId"], keep="last")
        chunk["rating_datetime"] = pd.to_datetime(chunk["timestamp"], unit="s")

        if write_full_ratings:
            chunk.to_csv(cleaned_path, index=False, mode="a", header=chunk_no == 1)

        part = chunk.groupby("movieId").agg(
            rating_count=("rating", "size"),
            rating_sum=("rating", "sum"),
            rating_mean=("rating", "mean"),
            first_rating_at=("rating_datetime", "min"),
            last_rating_at=("rating_datetime", "max"),
        )
        stats_parts.append(part.reset_index())
        print(f"Processed ratings chunk {chunk_no}: {rows_seen:,} rows")

    stats = pd.concat(stats_parts, ignore_index=True)
    stats = stats.groupby("movieId").agg(
        rating_count=("rating_count", "sum"),
        rating_sum=("rating_sum", "sum"),
        first_rating_at=("first_rating_at", "min"),
        last_rating_at=("last_rating_at", "max"),
    )
    stats["rating_mean"] = stats["rating_sum"] / stats["rating_count"]
    stats = stats.reset_index()
    stats["rating_mean"] = stats["rating_mean"].round(4)
    stats = stats[["movieId", "rating_count", "rating_mean", "first_rating_at", "last_rating_at"]]
    write_csv(stats, output_dir / "movie_rating_stats.csv")
    return stats


def clean_tags(tags_path: Path, output_dir: Path, sample_rows: int | None, chunksize: int) -> pd.DataFrame:
    tag_parts = []
    clean_path = output_dir / "tags_clean.csv"
    if clean_path.exists():
        clean_path.unlink()

    rows_seen = 0
    reader = pd.read_csv(tags_path, chunksize=chunksize)
    for chunk_no, chunk in enumerate(reader, start=1):
        if sample_rows is not None:
            remaining = sample_rows - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)

        chunk = chunk.dropna(subset=["userId", "movieId", "tag", "timestamp"]).copy()
        chunk["tag_clean"] = chunk["tag"].map(clean_text).str.lower()
        chunk = chunk[chunk["tag_clean"] != ""]
        chunk["tag_datetime"] = pd.to_datetime(chunk["timestamp"], unit="s")
        chunk = chunk[["userId", "movieId", "tag_clean", "tag_datetime"]].drop_duplicates()
        chunk.to_csv(clean_path, index=False, mode="a", header=chunk_no == 1)

        counts = chunk.groupby(["movieId", "tag_clean"]).size().reset_index(name="count")
        tag_parts.append(counts)
        print(f"Processed tags chunk {chunk_no}: {rows_seen:,} rows")

    tag_counts = pd.concat(tag_parts, ignore_index=True)
    tag_counts = tag_counts.groupby(["movieId", "tag_clean"], as_index=False)["count"].sum()
    tag_counts = tag_counts.sort_values(["movieId", "count", "tag_clean"], ascending=[True, False, True])

    top_tags = (
        tag_counts.groupby("movieId")
        .head(10)
        .groupby("movieId")["tag_clean"]
        .apply(lambda tags: "|".join(tags))
        .reset_index(name="user_tags_top10")
    )
    write_csv(top_tags, output_dir / "movie_user_tags_top10.csv")
    return top_tags


def clean_genome(
    genome_scores_path: Path,
    genome_tags_path: Path,
    output_dir: Path,
    sample_rows: int | None,
    relevance_threshold: float,
    chunksize: int,
) -> pd.DataFrame:
    genome_tags = pd.read_csv(genome_tags_path)
    genome_tags["tag"] = genome_tags["tag"].map(clean_text).str.lower()

    parts = []
    rows_seen = 0
    reader = pd.read_csv(genome_scores_path, chunksize=chunksize)
    for chunk_no, chunk in enumerate(reader, start=1):
        if sample_rows is not None:
            remaining = sample_rows - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)

        chunk = chunk[pd.to_numeric(chunk["relevance"], errors="coerce") >= relevance_threshold].copy()
        if not chunk.empty:
            parts.append(chunk)
        print(f"Processed genome chunk {chunk_no}: {rows_seen:,} rows")

    if parts:
        genome = pd.concat(parts, ignore_index=True)
        genome = genome.merge(genome_tags, on="tagId", how="left")
        genome = genome.sort_values(["movieId", "relevance"], ascending=[True, False])
        genome_top = (
            genome.groupby("movieId")
            .head(20)
            .groupby("movieId")["tag"]
            .apply(lambda tags: "|".join(tags.dropna().astype(str)))
            .reset_index(name="genome_tags_top20")
        )
    else:
        genome_top = pd.DataFrame(columns=["movieId", "genome_tags_top20"])

    write_csv(genome_top, output_dir / "movie_genome_tags_top20.csv")
    return genome_top


def build_master_table(
    movies: pd.DataFrame,
    links: pd.DataFrame,
    ratings: pd.DataFrame,
    user_tags: pd.DataFrame,
    genome_tags: pd.DataFrame,
    credits: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    master = movies.merge(links, on="movieId", how="left")
    master = master.merge(ratings, on="movieId", how="left")
    master = master.merge(user_tags, on="movieId", how="left")
    master = master.merge(genome_tags, on="movieId", how="left")
    master = master.merge(credits, on="tmdbId", how="left")

    text_cols = [
        "clean_title",
        "genres_list",
        "user_tags_top10",
        "genome_tags_top20",
        "top_cast",
        "director",
        "writers",
        "composer",
    ]
    for col in text_cols:
        if col in master.columns:
            master[col] = master[col].fillna("")

    master["content_features"] = (
        master["clean_title"].astype(str)
        + " "
        + master["genres_list"].astype(str)
        + " "
        + master["user_tags_top10"].astype(str).str.replace("|", " ", regex=False)
        + " "
        + master["genome_tags_top20"].astype(str).str.replace("|", " ", regex=False)
        + " "
        + master["top_cast"].astype(str).str.replace("|", " ", regex=False)
        + " "
        + master["director"].astype(str).str.replace("|", " ", regex=False)
        + " "
        + master["writers"].astype(str).str.replace("|", " ", regex=False)
    ).map(clean_text)

    master["rating_count"] = master["rating_count"].fillna(0).astype(int)
    master["rating_mean"] = master["rating_mean"].fillna(0).round(4)
    master = master.sort_values(["rating_count", "rating_mean"], ascending=[False, False])
    write_csv(master, output_dir / "movies_master.csv")
    return master


def write_data_dictionary(output_dir: Path, args: argparse.Namespace, outputs: dict[str, pd.DataFrame]) -> None:
    dictionary = {
        "project": "CINEIQ movie recommender dataset cleaning",
        "settings": {
            "sample_rows": args.sample_rows,
            "ratings_chunksize": args.chunksize,
            "write_full_ratings": args.write_full_ratings,
            "genome_relevance_threshold": args.genome_relevance_threshold,
        },
        "outputs": {
            name: {
                "rows": int(len(df)),
                "columns": list(df.columns),
            }
            for name, df in outputs.items()
        },
    }
    path = output_dir / "data_dictionary.json"
    path.write_text(json.dumps(dictionary, indent=2, default=str), encoding="utf-8")
    print(f"Wrote {path}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Clean CINEIQ movie recommendation datasets.")
    parser.add_argument("--base-dir", type=Path, default=DEFAULT_BASE)
    parser.add_argument("--ml25m-dir", type=Path, default=DEFAULT_ML25M)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT)
    parser.add_argument("--sample-rows", type=int, default=None, help="Read only N rows from large files for a quick test.")
    parser.add_argument("--chunksize", type=int, default=1_000_000)
    parser.add_argument("--write-full-ratings", action="store_true", help="Also write cleaned ratings_clean.csv.")
    parser.add_argument("--genome-relevance-threshold", type=float, default=0.70)
    # In Jupyter/IPython, the kernel injects arguments such as
    # "-f /path/to/kernel.json". parse_known_args keeps the script usable
    # both from Terminal and from a notebook cell.
    args, _unknown = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_args()
    output_dir = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    tmdb_path = args.base_dir / "tmdb_5000_credits.csv"
    imdb_path = args.base_dir / "IMDB Dataset.csv"
    movies_path = args.ml25m_dir / "movies.csv"
    links_path = args.ml25m_dir / "links.csv"
    ratings_path = args.ml25m_dir / "ratings.csv"
    tags_path = args.ml25m_dir / "tags.csv"
    genome_scores_path = args.ml25m_dir / "genome-scores.csv"
    genome_tags_path = args.ml25m_dir / "genome-tags.csv"

    movies = clean_movies(movies_path, output_dir)
    links = clean_links(links_path, output_dir)
    reviews = clean_imdb_reviews(imdb_path, output_dir, args.sample_rows)
    credits = clean_tmdb_credits(tmdb_path, output_dir, args.sample_rows)
    ratings = clean_ratings(ratings_path, output_dir, args.sample_rows, args.chunksize, args.write_full_ratings)
    user_tags = clean_tags(tags_path, output_dir, args.sample_rows, args.chunksize)
    genome_tags = clean_genome(
        genome_scores_path,
        genome_tags_path,
        output_dir,
        args.sample_rows,
        args.genome_relevance_threshold,
        args.chunksize,
    )
    master = build_master_table(movies, links, ratings, user_tags, genome_tags, credits, output_dir)

    write_data_dictionary(
        output_dir,
        args,
        {
            "movies_clean": movies,
            "links_clean": links,
            "imdb_reviews_clean": reviews,
            "tmdb_credits_clean": credits,
            "movie_rating_stats": ratings,
            "movie_user_tags_top10": user_tags,
            "movie_genome_tags_top20": genome_tags,
            "movies_master": master,
        },
    )

    print(f"\nDone. Main file for the recommender: {output_dir / 'movies_master.csv'}")


if __name__ == "__main__":
    main()


Wrote D:\projects\CineIQ\cleaned_data\movies_clean.csv (62,423 rows)
Wrote D:\projects\CineIQ\cleaned_data\links_clean.csv (62,423 rows)
Wrote D:\projects\CineIQ\cleaned_data\imdb_reviews_clean.csv (49,582 rows)
Wrote D:\projects\CineIQ\cleaned_data\tmdb_credits_clean.csv (4,803 rows)
Processed ratings chunk 1: 1,000,000 rows
Processed ratings chunk 2: 2,000,000 rows
Processed ratings chunk 3: 3,000,000 rows
Processed ratings chunk 4: 4,000,000 rows
Processed ratings chunk 5: 5,000,000 rows
Processed ratings chunk 6: 6,000,000 rows
Processed ratings chunk 7: 7,000,000 rows
Processed ratings chunk 8: 8,000,000 rows
Processed ratings chunk 9: 9,000,000 rows
Processed ratings chunk 10: 10,000,000 rows
Processed ratings chunk 11: 11,000,000 rows
Processed ratings chunk 12: 12,000,000 rows
Processed ratings chunk 13: 13,000,000 rows
Processed ratings chunk 14: 14,000,000 rows
Processed ratings chunk 15: 15,000,000 rows
Processed ratings chunk 16: 16,000,000 rows
Processed ratings chunk 17: 

In [2]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = resolve_project_root()
movies = pd.read_csv(PROJECT_ROOT / "cleaned_data" / "movies_master.csv", low_memory=False)
movies.head(10)



,movieId,title,clean_title,year,genres,genres_list,primary_genre,genre_count,imdbId,tmdbId,...,user_tags_top10,genome_tags_top20,tmdb_title,top_cast,cast_count,director,writers,producers,composer,content_features
0,356,Forrest Gump (1994),Forrest Gump,1994.0,Comedy|Drama|Romance|War,Comedy Drama Romance War,Comedy,4,109830,13.0,...,tom hanks|inspirational|classic|bittersweet|co...,oscar (best music - original score)|oscar (bes...,Forrest Gump,Tom Hanks|Robin Wright|Gary Sinise|Mykelti Wil...,66.0,Robert Zemeckis,Eric Roth,Wendy Finerman|Steve Tisch|Steve Starkey,Alan Silvestri,Forrest Gump Comedy Drama Romance War tom hank...
1,318,"Shawshank Redemption, The (1994)","Shawshank Redemption, The",1994.0,Crime|Drama,Crime Drama,Crime,2,111161,278.0,...,morgan freeman|prison|prison escape|twist endi...,imdb top 250|oscar (best picture)|powerful end...,The Shawshank Redemption,Tim Robbins|Morgan Freeman|Bob Gunton|Clancy B...,42.0,Frank Darabont,Frank Darabont,Niki Marvin,Thomas Newman,"Shawshank Redemption, The Crime Drama morgan f..."
2,296,Pulp Fiction (1994),Pulp Fiction,1994.0,Comedy|Crime|Drama|Thriller,Comedy Crime Drama Thriller,Comedy,4,110912,680.0,...,quentin tarantino|dark comedy|nonlinear|multip...,hit men|gratuitous violence|dark humor|masterp...,Pulp Fiction,John Travolta|Samuel L. Jackson|Uma Thurman|Br...,54.0,Quentin Tarantino,Quentin Tarantino|Roger Avary,Lawrence Bender,NaN,Pulp Fiction Comedy Crime Drama Thriller quent...
3,593,"Silence of the Lambs, The (1991)","Silence of the Lambs, The",1991.0,Crime|Horror|Thriller,Crime Horror Thriller,Crime,3,102926,274.0,...,serial killer|psychology|anthony hopkins|suspe...,oscar (best directing)|serial killer|suspensef...,The Silence of the Lambs,Jodie Foster|Anthony Hopkins|Scott Glenn|Ted L...,59.0,Jonathan Demme,Ted Tally,Ronald M. Bozman|Edward Saxon|Kenneth Utt,Howard Shore,"Silence of the Lambs, The Crime Horror Thrille..."
4,2571,"Matrix, The (1999)","Matrix, The",1999.0,Action|Sci-Fi|Thriller,Action Sci-Fi Thriller,Action,3,133093,603.0,...,sci-fi|virtual reality|dystopia|philosophy|cyb...,dystopic future|scifi|cyberpunk|sci fi|science...,The Matrix,Keanu Reeves|Laurence Fishburne|Carrie-Anne Mo...,36.0,Lilly Wachowski|Lana Wachowski,Lilly Wachowski|Lana Wachowski,Joel Silver,Don Davis,"Matrix, The Action Sci-Fi Thriller sci-fi virt..."
5,260,Star Wars: Episode IV - A New Hope (1977),Star Wars: Episode IV - A New Hope,1977.0,Action|Adventure|Sci-Fi,Action Adventure Sci-Fi,Action,3,76759,11.0,...,sci-fi|space|classic|science fiction|space adv...,space opera|science fiction|sci fi|scifi|trilo...,Star Wars,Mark Hamill|Harrison Ford|Carrie Fisher|Peter ...,106.0,George Lucas,George Lucas,Gary Kurtz|Rick McCallum,John Williams,Star Wars: Episode IV - A New Hope Action Adve...
6,480,Jurassic Park (1993),Jurassic Park,1993.0,Action|Adventure|Sci-Fi|Thriller,Action Adventure Sci-Fi Thriller,Action,4,107290,329.0,...,dinosaurs|steven spielberg|adventure|sci-fi|ge...,special effects|dinosaurs|spielberg|big budget...,Jurassic Park,Sam Neill|Laura Dern|Jeff Goldblum|Richard Att...,27.0,Steven Spielberg,David Koepp|Michael Crichton,Kathleen Kennedy|Gerald R. Molen,John Williams,Jurassic Park Action Adventure Sci-Fi Thriller...
7,527,Schindler's List (1993),Schindler's List,1993.0,Drama|War,Drama War,Drama,2,108052,424.0,...,holocaust|world war ii|true story|steven spiel...,holocaust|jews|oscar (best directing)|nazis|wo...,Schindler's List,Liam Neeson|Ben Kingsley|Ralph Fiennes|Carolin...,29.0,Steven Spielberg,Steven Zaillian,Steven Spielberg|Branko Lustig|Gerald R. Molen,John Williams,Schindler's List Drama War holocaust world war...
8,110,Braveheart (1995),Braveheart,1995.0,Action|Drama|War,Action Drama War,Action,3,112573,197.0,...,mel gibson|historical|medieval|war|action|insp...,historical|history|oscar (best picture)|oscar ...,Braveheart,Mel Gibson|Catherine McCormack|Sophie Marceau|...,55.0,Mel Gibson,Randall Wallace,"Mel Gibson|Bruce Davey|Alan Ladd, Jr.|Elisabet...",Ja

In [3]:
from pathlib import Path

PROJECT_ROOT = resolve_project_root()
destination = PROJECT_ROOT / "cleaned_data"

if not destination.exists():
    raise FileNotFoundError("cleaned_data folder not found. Run the cleaning script first.")

print(f"Cleaned data is stored inside this project at: {destination}")


Cleaned data is stored inside this project at: D:\projects\CineIQ\cleaned_data


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = resolve_project_root()
data_path = PROJECT_ROOT / "cleaned_data"
movies = pd.read_csv(data_path / "movies_master.csv", low_memory=False)

movies["content_features"] = movies["content_features"].fillna("")
movies["clean_title"] = movies["clean_title"].fillna("")
movies["rating_count"] = pd.to_numeric(movies["rating_count"], errors="coerce").fillna(0)
movies["rating_mean"] = pd.to_numeric(movies["rating_mean"], errors="coerce").fillna(0)

movies = movies[movies["content_features"].str.strip() != ""].copy()
movies.reset_index(drop=True, inplace=True)

print(movies.shape)


(62423, 25)


In [5]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(movies["content_features"])
print(tfidf_matrix.shape)


(62423, 10000)


In [6]:
nn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=20)
nn_model.fit(tfidf_matrix)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",20
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [7]:
indices = pd.Series(movies.index, index=movies["clean_title"].str.lower()).drop_duplicates()


In [8]:
def recommend_movies(title, top_n=10):
    title = title.lower().strip()

    if title not in indices:
        return f"Movie '{title}' not found in dataset."

    idx = indices[title]

    distances, neighbors = nn_model.kneighbors(tfidf_matrix[idx], n_neighbors=top_n + 1)

    rec_indices = neighbors.flatten()[1:]
    rec_distances = distances.flatten()[1:]

    recs = movies.iloc[rec_indices][[
        "movieId",
        "clean_title",
        "year",
        "genres",
        "rating_count",
        "rating_mean",
        "director",
        "top_cast"
    ]].copy()

    recs["similarity"] = 1 - rec_distances
    recs = recs.sort_values(["similarity", "rating_count"], ascending=[False, False])

    return recs.reset_index(drop=True)


In [9]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 1)
)


In [10]:
recommend_movies("toy story", top_n=10)
recommend_movies("matrix, the", top_n=10)
recommend_movies("batman", top_n=10)
recommend_movies("finding nemo", top_n=10)


,movieId,clean_title,year,genres,rating_count,rating_mean,director,top_cast,similarity
0,157296,Finding Dory,2016.0,Adventure|Animation|Comedy,3310,3.6156,NaN,NaN,0.618530
1,2355,"Bug's Life, A",1998.0,Adventure|Animation|Children|Comedy,22471,3.5692,Andrew Stanton|John Lasseter,Kevin Spacey|Julia Louis-Dreyfus|Hayden Panett...,0.530244
2,4886,"Monsters, Inc.",2001.0,Adventure|Animation|Children|Comedy|Fantasy,34572,3.8486,Pete Docter,John Goodman|Billy Crystal|Mary Gibbs|Steve Bu...,0.474279
3,50872,Ratatouille,2007.0,Animation|Children|Drama,19157,3.8111,Jan Pinkava|Brad Bird,Patton Oswalt|Ian Holm|Lou Romano|Brian Denneh...,0.473720
4,95375,Boundin',2003.0,Animation|Children,218,3.4679,NaN,NaN,0.443455
5,1,Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy,57309,3.8937,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,0.423341
6,109425,Dug's Special Mission,2009.0,Animation|Children|Comedy,86,3.3663,NaN,NaN,0.418130
7,167036,Sing,2016.0,Animation|Children|Comedy,886,3.4554,NaN,NaN,0.415844
8,60069,WALL·E,2008.0,Adventure|Animation|Children|Romance|Sci-Fi,27374,4.0049,Andrew Stanton,Ben Burtt|Elissa Knight|Jeff Garlin|Fred Willa...,0.408185
9,68954,Up,2009.0,Adventure|Animation|Children|Drama,25127,3.9636,Pete Docter,Ed Asner|Christopher Plummer|Jordan Nagai|Bob ...,0.406015


In [11]:
recommend_movies("interstellar", top_n=10)


,movieId,clean_title,year,genres,rating_count,rating_mean,director,top_cast,similarity
0,91529,"Dark Knight Rises, The",2012.0,Action|Adventure|Crime|IMAX,19912,3.9713,Christopher Nolan,Christian Bale|Michael Caine|Gary Oldman|Anne ...,0.450134
1,103306,Europa Report,2013.0,Sci-Fi|Thriller,869,3.3861,NaN,NaN,0.420550
2,159972,Approaching the Unknown,2016.0,Drama|Sci-Fi|Thriller,54,2.3889,NaN,NaN,0.414098
3,104841,Gravity,2013.0,Action|Sci-Fi|IMAX,12264,3.6170,Alfonso Cuarón,Sandra Bullock|George Clooney|Ed Harris|Orto I...,0.391318
4,1584,Contact,1997.0,Drama|Sci-Fi,21638,3.6845,Robert Zemeckis,Jodie Foster|Matthew McConaughey|James Woods|J...,0.380855
5,166635,Passengers,2016.0,Adventure|Drama|Romance|Sci-Fi,3171,3.5396,NaN,NaN,0.380555
6,3687,Light Years (Gandahar),1988.0,Adventure|Animation|Fantasy|Sci-Fi,87,3.3103,NaN,NaN,0.380107
7,134130,The Martian,2015.0,Adventure|Drama|Sci-Fi,16489,4.0334,Ridley Scott,Matt Damon|Jessica Chastain|Kristen Wiig|Jeff ...,0.378577
8,71106,Frequently Asked Questions About Time Travel,2009.0,Comedy|Sci-Fi,743,3.7678,NaN,NaN,0.378506
9,3780,Rocketship X-M,1950.0,Sci-Fi,121,2.6033,NaN,NaN,0.377877


In [12]:
import pickle
from pathlib import Path

PROJECT_ROOT = resolve_project_root()
artifacts = PROJECT_ROOT / "model_artifacts"
artifacts.mkdir(parents=True, exist_ok=True)

with (artifacts / "tfidf_vectorizer.pkl").open("wb") as file_handle:
    pickle.dump(tfidf, file_handle)

with (artifacts / "nn_model.pkl").open("wb") as file_handle:
    pickle.dump(nn_model, file_handle)

movies.to_csv(artifacts / "movies_ready.csv", index=False)

print(f"Saved successfully to: {artifacts}")


Saved successfully to: D:\projects\CineIQ\model_artifacts


In [13]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = resolve_project_root()
data_path = PROJECT_ROOT / "cleaned_data"
movies = pd.read_csv(data_path / "movies_master.csv", low_memory=False)






In [14]:
import random

import pandas as pd
from pathlib import Path

PROJECT_ROOT = resolve_project_root()
data_path = PROJECT_ROOT / "cleaned_data"
raw_data_path = PROJECT_ROOT / "raw_data" / "ml-25m"
movies = pd.read_csv(data_path / "movies_master.csv", low_memory=False)

ratings_path = raw_data_path / "ratings.csv"
ratings_sample_fraction = 0.024
random.seed(42)
rng = random.Random(42)

ratings = pd.read_csv(
    ratings_path,
    skiprows=lambda row_number: row_number > 0 and rng.random() >= ratings_sample_fraction,
)

print(movies.shape)
print(f"Loaded unbiased ratings sample with seed 42: {ratings.shape}")
ratings.head()


(62423, 25)
Loaded unbiased ratings sample with seed 42: (598138, 4)


,userId,movieId,rating,timestamp
0,1,2692,5.0,1147869100
1,2,1080,1.0,1141415532
2,2,1485,3.0,1141415592
3,2,1527,3.0,1141417716
4,3,527,4.0,1439472436


In [15]:
user_counts = ratings["userId"].value_counts()
movie_counts = ratings["movieId"].value_counts()

active_users = user_counts[user_counts >= 20].index
popular_movies = movie_counts[movie_counts >= 50].index

ratings_small = ratings[
    ratings["userId"].isin(active_users) &
    ratings["movieId"].isin(popular_movies)
].copy()

print(ratings_small.shape)


(88508, 4)


In [16]:
user_movie_matrix = ratings_small.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print(user_movie_matrix.shape)


(4354, 2414)


In [17]:
movie_similarity = user_movie_matrix.corr(method="pearson", min_periods=20)
print(movie_similarity.shape)


(2414, 2414)


In [18]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = resolve_project_root()
movies_df = pd.read_csv(PROJECT_ROOT / "cleaned_data" / "movies_master.csv", low_memory=False)

movie_id_to_title = dict(zip(movies_df["movieId"], movies_df["clean_title"]))
title_to_movie_id = dict(zip(movies_df["clean_title"].str.lower(), movies_df["movieId"]))


In [19]:
def recommend_collaborative(movie_title, top_n=10):
    movie_title = movie_title.lower().strip()
    
    if movie_title not in title_to_movie_id:
        return f"Movie '{movie_title}' not found."
    
    movie_id = title_to_movie_id[movie_title]
    
    if movie_id not in movie_similarity.columns:
        return f"No collaborative recommendations available for '{movie_title}'."
    
    sim_scores = movie_similarity[movie_id].dropna().sort_values(ascending=False)
    sim_scores = sim_scores.iloc[1:top_n+1]
    
    result = pd.DataFrame({
        "movieId": sim_scores.index,
        "similarity": sim_scores.values
    })
    
    result["title"] = result["movieId"].map(movie_id_to_title)
    return result[["movieId", "title", "similarity"]]


In [20]:
recommend_collaborative("Avengers: Infinity War - Part II", top_n=10)

,movieId,title,similarity


In [21]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = resolve_project_root()
movies = pd.read_csv(PROJECT_ROOT / "cleaned_data" / "movies_master.csv", low_memory=False)
movies["top_cast"] = movies["top_cast"].fillna("")

def search_by_cast(actor_name):
    actor_name = actor_name.lower().strip()
    
    results = movies[movies["top_cast"].str.lower().str.contains(actor_name, na=False)].copy()
    
    return results[[
        "movieId", "clean_title", "year", "genres", "top_cast", "director", "rating_mean"
    ]].sort_values(by="rating_mean", ascending=False)

search_by_cast("tom hanks").head(20)


,movieId,clean_title,year,genres,top_cast,director,rating_mean
0,356,Forrest Gump,1994.0,Comedy|Drama|Romance|War,Tom Hanks|Robin Wright|Gary Sinise|Mykelti Wil...,Robert Zemeckis,4.0480
28,2028,Saving Private Ryan,1998.0,Action|Drama|War,Tom Hanks|Matt Damon|Vin Diesel|Tom Sizemore|B...,Steven Spielberg,4.0441
88,3147,"Green Mile, The",1999.0,Crime|Drama,Tom Hanks|Michael Clarke Duncan|David Morse|Bo...,Frank Darabont,4.0278
104,5989,Catch Me If You Can,2002.0,Crime|Drama,Leonardo DiCaprio|Tom Hanks|Christopher Walken...,Steven Spielberg,3.9290
12,1,Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,John Lasseter,3.8937
25,150,Apollo 13,1995.0,Adventure|Drama|IMAX,Tom Hanks|Bill Paxton|Kevin Bacon|Gary Sinise|...,Ron Howard,3.8736
336,78499,Toy Story 3,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,Tom Hanks|Tim Allen|Ned Beatty|Joan Cusack|Mic...,Lee Unkrich,3.8578
118,3114,Toy Story 2,1999.0,Adventure|Animation|Children|Comedy|Fantasy,Tom Hanks|Tim Allen|Joan Cusack|Kelsey Grammer...,John Lasseter,3.8115
204,508,Philadelphia,1993.0,Drama,Tom Hanks|Denzel Washington|Jason Robards|Mary...,Jonathan Demme,3.8021
1280,105504,Captain Phillips,2013.0,Adventure|Drama|Thriller|IMAX,Tom Hanks|Catherine Keener|Max Martini|Chris M...,Paul Greengrass,3.7938


In [22]:
def recommend_by_cast(actor_name, top_n=10):
    actor_name = actor_name.lower().strip()
    
    results = movies[movies["top_cast"].str.lower().str.contains(actor_name, na=False)].copy()
    
    if results.empty:
        return f"No movies found for cast member '{actor_name}'."
    
    results = results.sort_values(
        by=["rating_mean", "rating_count"],
        ascending=False
    ).head(top_n)
    
    return results[[
        "movieId", "clean_title", "year", "genres", "top_cast", "director", "rating_mean", "rating_count"
    ]]

recommend_by_cast("tom hanks", top_n=10)


,movieId,clean_title,year,genres,top_cast,director,rating_mean,rating_count
0,356,Forrest Gump,1994.0,Comedy|Drama|Romance|War,Tom Hanks|Robin Wright|Gary Sinise|Mykelti Wil...,Robert Zemeckis,4.0480,81491
28,2028,Saving Private Ryan,1998.0,Action|Drama|War,Tom Hanks|Matt Damon|Vin Diesel|Tom Sizemore|B...,Steven Spielberg,4.0441,46783
88,3147,"Green Mile, The",1999.0,Crime|Drama,Tom Hanks|Michael Clarke Duncan|David Morse|Bo...,Frank Darabont,4.0278,30482
104,5989,Catch Me If You Can,2002.0,Crime|Drama,Leonardo DiCaprio|Tom Hanks|Christopher Walken...,Steven Spielberg,3.9290,27935
12,1,Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,John Lasseter,3.8937,57309
25,150,Apollo 13,1995.0,Adventure|Drama|IMAX,Tom Hanks|Bill Paxton|Kevin Bacon|Gary Sinise|...,Ron Howard,3.8736,48377
336,78499,Toy Story 3,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,Tom Hanks|Tim Allen|Ned Beatty|Joan Cusack|Mic...,Lee Unkrich,3.8578,14426
118,3114,Toy Story 2,1999.0,Adventure|Animation|Children|Comedy|Fantasy,Tom Hanks|Tim Allen|Joan Cusack|Kelsey Grammer...,John Lasseter,3.8115,26536
204,508,Philadelphia,1993.0,Drama,Tom Hanks|Denzel Washington|Jason Robards|Mary...,Jonathan Demme,3.8021,19902
1280,105504,Captain Phillips,2013.0,Adventure|Drama|Thriller|IMAX,Tom Hanks|Catherine Keener|Max Martini|Chris M...,Paul Greengrass,3.7938,4744


In [23]:
movies["director"] = movies["director"].fillna("")

def search_by_director(name):
    name = name.lower().strip()
    results = movies[movies["director"].str.lower().str.contains(name, na=False)].copy()
    return results[[
        "movieId", "clean_title", "year", "genres", "director", "top_cast", "rating_mean"
    ]].sort_values(by="rating_mean", ascending=False)

search_by_director("christopher nolan").head(20)


,movieId,clean_title,year,genres,director,top_cast,rating_mean
37,58559,"Dark Knight, The",2008.0,Action|Crime|Drama|IMAX,Christopher Nolan,Christian Bale|Heath Ledger|Aaron Eckhart|Mich...,4.1665
42,79132,Inception,2010.0,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,Christopher Nolan,Leonardo DiCaprio|Joseph Gordon-Levitt|Ellen P...,4.1555
39,4226,Memento,2000.0,Mystery|Thriller,Christopher Nolan,Guy Pearce|Carrie-Anne Moss|Joe Pantoliano|Mar...,4.1500
163,109487,Interstellar,2014.0,Sci-Fi|IMAX,Christopher Nolan,Matthew McConaughey|Jessica Chastain|Anne Hath...,4.0979
159,48780,"Prestige, The",2006.0,Drama|Mystery|Sci-Fi|Thriller,Christopher Nolan,Hugh Jackman|Christian Bale|Michael Caine|Scar...,4.0932
203,91529,"Dark Knight Rises, The",2012.0,Action|Adventure|Crime|IMAX,Christopher Nolan,Christian Bale|Michael Caine|Gary Oldman|Anne ...,3.9713
86,33794,Batman Begins,2005.0,Action|Crime|IMAX,Christopher Nolan,Christian Bale|Michael Caine|Liam Neeson|Katie...,3.9300
805,5388,Insomnia,2002.0,Action|Crime|Drama|Mystery|Thriller,Christopher Nolan,Al Pacino|Robin Williams|Hilary Swank|Maura Ti...,3.4835


In [24]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = resolve_project_root()
movies = pd.read_csv(PROJECT_ROOT / "cleaned_data" / "movies_master.csv", low_memory=False)

movies["clean_title"] = movies["clean_title"].fillna("")
movies["top_cast"] = movies["top_cast"].fillna("")
movies["director"] = movies["director"].fillna("")
movies["genres"] = movies["genres"].fillna("")

def search_movies(query):
    query = query.lower().strip()

    results = movies[
        movies["clean_title"].str.lower().str.contains(query, na=False) |
        movies["top_cast"].str.lower().str.contains(query, na=False) |
        movies["director"].str.lower().str.contains(query, na=False) |
        movies["genres"].str.lower().str.contains(query, na=False)
    ].copy()

    return results[[
        "movieId", "clean_title", "year", "genres",
        "top_cast", "director", "rating_mean", "rating_count"
    ]].sort_values(
        by=["rating_mean", "rating_count"],
        ascending=False
    )

search_movies("sci-fi").head(20)


,movieId,clean_title,year,genres,top_cast,director,rating_mean,rating_count
36194,148298,Awaken,2013.0,Drama|Romance|Sci-Fi,,,5.0,3
41136,182657,Pale,2016.0,Sci-Fi|Thriller,,,5.0,2
48839,132124,Say Nothing,2001.0,Action|Drama|Mystery|Romance|Sci-Fi|Thriller,,,5.0,1
48906,137733,Lost Time,2014.0,Horror|Sci-Fi|Thriller,,,5.0,1
48918,138280,Plato's Reality Machine,2013.0,Comedy|Drama|Sci-Fi,,,5.0,1
48939,139473,Lost Woods,2012.0,Action|Horror|Sci-Fi|Thriller,,,5.0,1
48990,145330,Unidentified,2006.0,Sci-Fi,,,5.0,1
49013,148278,"10,000 Days",2014.0,Sci-Fi,,,5.0,1
49077,160008,Moonchild,1974.0,Drama|Horror|Sci-Fi,,,5.0,1
49090,161902,Terror of Frankenstein,1977.0,Horror|Sci-Fi,,,5.0,1


In [25]:
search_movies("christopher nolan").head(10)


,movieId,clean_title,year,genres,top_cast,director,rating_mean,rating_count
37,58559,"Dark Knight, The",2008.0,Action|Crime|Drama|IMAX,Christian Bale|Heath Ledger|Aaron Eckhart|Mich...,Christopher Nolan,4.1665,41519
42,79132,Inception,2010.0,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,Leonardo DiCaprio|Joseph Gordon-Levitt|Ellen P...,Christopher Nolan,4.1555,38895
39,4226,Memento,2000.0,Mystery|Thriller,Guy Pearce|Carrie-Anne Moss|Joe Pantoliano|Mar...,Christopher Nolan,4.1500,41195
163,109487,Interstellar,2014.0,Sci-Fi|IMAX,Matthew McConaughey|Jessica Chastain|Anne Hath...,Christopher Nolan,4.0979,22634
159,48780,"Prestige, The",2006.0,Drama|Mystery|Sci-Fi|Thriller,Hugh Jackman|Christian Bale|Michael Caine|Scar...,Christopher Nolan,4.0932,22943
203,91529,"Dark Knight Rises, The",2012.0,Action|Adventure|Crime|IMAX,Christian Bale|Michael Caine|Gary Oldman|Anne ...,Christopher Nolan,3.9713,19912
86,33794,Batman Begins,2005.0,Action|Crime|IMAX,Christian Bale|Michael Caine|Liam Neeson|Katie...,Christopher Nolan,3.9300,30684
805,5388,Insomnia,2002.0,Action|Crime|Drama|Mystery|Thriller,Al Pacino|Robin Williams|Hilary Swank|Maura Ti...,Christopher Nolan,3.4835,7591
